# Figure 4 — distance between the crossings and the model bow shock

Rebuilds the three-panel distance figure for the revision, from the same OMNI cache, the same
crossing list and the same boundary model as `render_passes_v2.ipynb`, so the figure and the
renderings cannot disagree.

Run top to bottom. Cell 5 prints every number the reply letter needs, including both of the open
choices below, so you can see the effect before deciding.

**Both choices below were settled on 3 October: `min_surface` and `crossing`, which are the
defaults.** They stay switchable because the comparison is worth being able to reproduce, and cell 4
still prints the figure both ways.

`DISTANCE_DEF`
- `"fixed_x"` is the definition in the *submitted* Figure 4 caption,
  $d = |\rho_{\rm obs} - \rho_{\rm model}(x_{\rm obs})|$, measured straight out from the $x$ axis.
  It is undefined for a crossing that lies sunward of the model nose, and 41 crossings do.
  Dropping them removes exactly the cases where the model most underestimates the shock.
- `"min_surface"` is the shortest distance from the crossing to the model surface in the
  $(x, \rho)$ plane, signed positive outside. Defined for every crossing, keeps the 41, and is the
  usual meaning of "distance from the boundary". Choosing this needs one sentence of the caption
  rewritten.

`PARAM_SOURCE`
- `"crossing"` evaluates the model at each crossing from the nearest valid OMNI hour to that
  crossing. This is what Section 4 of the manuscript already claims is done.
- `"leg"` uses the single parameter set per pass leg from `pass_parameters_v2.csv`, which is what
  the 3D renderings were drawn with. The two differ because a leg can span many hours: on pass 47
  inbound the Alfvén Mach number runs from 1.5 to 3.2 across the leg, while the rendering used 1.40.

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import json

# Machine-specific locations come from local_paths.json beside this notebook. That file is not part of the
# repository: copy local_paths.example.json and fill it in. Without it, BOW_SHOCK is the folder above this one.
_local_file = Path("local_paths.json")
LOCAL = json.loads(_local_file.read_text(encoding="utf-8")) if _local_file.exists() else {}
BOW_SHOCK = Path(LOCAL.get("bow_shock", Path.cwd().parent))
MAGCARTO  = BOW_SHOCK / "MagCarto"

OMNI_CACHE = Path(LOCAL.get("omni_cache", BOW_SHOCK / "OMNI" / "omni2_cache.pkl"))
CROSSINGS  = BOW_SHOCK / "bs_crossings_V02.xlsx"
EPHEM      = BOW_SHOCK / "Bow_Shock_Stats" / "Wind_Ephemerids.h5"
EPHEM_KEY  = "/GSM2"
PARAM_CSV  = MAGCARTO / "pass_parameters_v2.csv"

OUT_FIG = MAGCARTO / "distance_v2.png"
OUT_CSV = MAGCARTO / "crossing_distances_v2.csv"
# To write straight into the paper's LaTeX project instead, point OUT_FIG at
# Path(LOCAL["overleaf"]) / "distance.png"   (the "overleaf" entry of local_paths.json)

RE_KM = 6371.0

# --- the two open choices ----------------------------------------------------------------------
DISTANCE_DEF  = "min_surface"   # "min_surface" or "fixed_x"
PARAM_SOURCE  = "crossing"      # "crossing" or "leg"

# --- selection ----------------------------------------------------------------------------------
X_MIN         = 0.0    # keep crossings with x_GSM >= this; the model is not meant for the tail
OMNI_WINDOW_H = 4.0    # same window as the rendering pipeline

# --- model ----------------------------------------------------------------------------------
GAMMA     = 5.0 / 3.0
BS_ECCENT = 0.9
N_THETA   = 4000       # sampling of the conic when minimising the distance

# --- binning ----------------------------------------------------------------------------------
AZIMUTH_EDGES = np.arange(-180, 181, 30)   # 12 bins of 30 deg
MIN_BIN_COUNT = 1                          # a bin with fewer than this is drawn as empty
N_PRESSURE_BINS = 8                        # equal-count, as Figure 5 bins in Mach number

# --- drawing ------------------------------------------------------------------------------------
FIG_W, FIG_H, FIG_DPI = 14.2, 9.4, 200
PANEL_WIDTHS, CBAR_WIDTH   = (1.0, 1.22), 0.045
PANEL_WSPACE, PANEL_HSPACE = 0.33, 0.26

HIST_BINS       = np.arange(0, 14.1, 0.5)
HIST_FACE       = "#4878a8"
HIST_EDGE       = "white"
HIST_EDGEWIDTH  = 0.6
HIST_ALPHA      = 0.95

MEDIAN_LINE_COLOR, MEDIAN_LINE_STYLE, MEDIAN_LINE_WIDTH = "#c44e52", "--", 1.6

IQR_COLOR, IQR_MARKER, IQR_MARKERSIZE = "#2c3e50", "o", 5
IQR_LINEWIDTH, IQR_CAPSIZE            = 1.6, 3

ANNOT_FONTSIZE, ANNOT_COLOR        = 7.5, "#2c3e50"
EMPTY_FONTSIZE, EMPTY_COLOR        = 8.0, "#999999"

SCATTER_CMAP, SCATTER_SIZE   = "viridis", 22
SCATTER_ALPHA, SCATTER_EDGE  = 0.85, "white"
SCATTER_EDGEWIDTH            = 0.4

OUTLIER_COLOR, OUTLIER_SIZE, OUTLIER_MARKER = "#c44e52", 70, "x"
OUTLIER_LINEWIDTH   = 1.8             # the marker carries no label; the caption identifies it
SCATTER_YTICKS      = (0, 1, 2, 3, 4, 5, 6)   # panel (c), below the cut
SCATTER_Y_TOP       = 7.30            # top of panel (c), once the scale is cut
BREAK_LOW, BREAK_HIGH, BREAK_TILT = 6.50, 6.75, 0.10   # where the scale of panel (c) jumps
BREAK_FACECOLOR, BREAK_COLOR, BREAK_WIDTH = "white", "#2c3e50", 1.0
OUTLIER_DRAWN_AT    = 7.05            # where the one crossing above the cut is drawn
AZIMUTH_ANNOT_HEADROOM = 1.45         # top of panel (b), above its annotation row

MODEL_COLOR, MODEL_STYLE, MODEL_WIDTH = "#d99c2b", "-", 1.8
MODEL_MARKER, MODEL_MARKERSIZE        = "s", 4.5

PRESSURE_XSCALE, PRESSURE_YSCALE = "log", "log"
PRESSURE_X_LOW, PRESSURE_X_HIGH  = 0.92, 1.08   # edges of panel (d), as multiples of the pressure
PRESSURE_Y_LOW, PRESSURE_Y_HIGH  = 0.90, 1.40   # and of the implied standoff
PRESSURE_XTICKS      = (0.3, 0.5, 1, 2, 3, 5, 10)   # plain decimals, not powers of ten
PRESSURE_ANNOT_YFRAC = (0.925, 0.835)   # two annotation rows; the bins alternate between them
PRESSURE_LEGEND_LOC  = "lower left"

PANEL_LABELS = ("(a)", "(b)", "(c)", "(d)")
PANEL_LABEL_SIZE, PANEL_LABEL_COLOR = 12, "black"
PANEL_LABEL_X, PANEL_LABEL_Y        = -0.10, 1.04

LABEL_SIZE, TICK_SIZE, TITLE_SIZE = 11, 9.5, 11
GRID_COLOR, GRID_ALPHA, GRID_WIDTH = "#cccccc", 0.6, 0.6

for p in (OMNI_CACHE, CROSSINGS, EPHEM, PARAM_CSV):
    print(f"{'ok ' if p.exists() else 'MISSING':<8} {p}")
print(f"{'->':<8} {OUT_FIG}")
print(f"{'->':<8} {OUT_CSV}")
print(f"\ndistance = {DISTANCE_DEF}, parameters per {PARAM_SOURCE}")

## 1. Load everything, exactly as the rendering notebook does

In [ ]:
cache = pickle.load(open(OMNI_CACHE, "rb"))
FIELDS = ["Flow_pressure", "Dst_index", "By_GSM", "Bz_GSM", "Alfven_mach_number"]
omni = pd.DataFrame({c: np.asarray(cache[c], dtype=float) for c in FIELDS})
omni["Epoch"] = pd.to_datetime(list(cache["Epoch"]))
omni = omni.dropna(subset=FIELDS).sort_values("Epoch").reset_index(drop=True)
_omni_epoch = omni.Epoch.values
print(f"OMNI: {len(omni)} hours with all five fields valid")

store = pd.HDFStore(EPHEM, "r")
eph = store.select(EPHEM_KEY)
store.close()
eph.columns = [c[0] if isinstance(c, tuple) else c for c in eph.columns]
eph = eph.sort_index()
print(f"ephemeris: {len(eph)} records, step {pd.Series(eph.index).diff().median()}")

x = pd.read_excel(CROSSINGS, sheet_name=0)
for c in ("time", "start time", "end time", "start time zoom", "end time zoom"):
    x[c] = pd.to_datetime(x[c])
# A leg is a run of consecutive rows with the same pass and SC direction. Until October 2026 the window
# columns were filled only on the first row of each leg and legs were found from the blank cells; the
# windows are now repeated on every row, so the leg is taken from pass and direction instead. On the
# old file both rules give the same 140 legs.
_key = x[["pass", "SC direction"]]
x["leg"] = (_key != _key.shift()).any(axis=1).cumsum()
assert x.groupby(["pass", "SC direction"])["leg"].nunique().max() == 1, \
    "a (pass, SC direction) pair is split into more than one block of rows"
print(f"crossing list: {len(x)} crossings in {x.leg.nunique()} pass legs")

par = pd.read_csv(PARAM_CSV)
par_ok = par[par.status == "ok"].set_index(["pass_id", "sc_direction"])
print(f"per-leg parameters: {len(par)} legs, {len(par_ok)} rendered, "
      f"{len(par) - len(par_ok)} skipped")

## 2. The boundary model

Identical to the rendering pipeline. The only new piece is the inversion of the conic.

The bow shock is $r(\theta) = L / (1 + e\cos\theta)$ with $L = r_{bs0}(1+e)$. Since $x = r\cos\theta$,
that rearranges to $r = L - e x$ exactly, so with $r^2 = x^2 + \rho^2$

$$\rho_{\rm model}(x) = \sqrt{(L - e x)^2 - x^2},$$

which is real only for $x \le L/(1+e) = r_{bs0}$, i.e. not sunward of the nose. That is the hole in
the `"fixed_x"` definition.

In [ ]:
def shue98_r0_alpha(pdyn_npa, bz_nt):
    """Shue et al. (1998) Eqs. 10 and 11."""
    r0 = (10.22 + 1.29 * np.tanh(0.184 * (bz_nt + 8.14))) * pdyn_npa ** (-1.0 / 6.6)
    alpha = (0.58 - 0.007 * bz_nt) * (1.0 + 0.024 * np.log(pdyn_npa))
    return r0, alpha


def standoff(r0_mp, mach, gamma=GAMMA):
    """Gas-dynamic standoff of the shock above the magnetopause nose. Unclamped."""
    return r0_mp * ((gamma - 1.0) * mach**2 + 2.0) / ((gamma + 1.0) * (mach**2 - 1.0))


def bowshock_rho_at_x(x_re, r_bs0, ecc=BS_ECCENT):
    """Cylindrical radius of the model surface at x. NaN sunward of the nose."""
    L = r_bs0 * (1.0 + ecc)
    q = (L - ecc * x_re) ** 2 - x_re ** 2
    return np.sqrt(q) if q > 0 else np.nan


_theta = np.linspace(1e-4, np.pi * 0.995, N_THETA)


def bowshock_min_distance(x_re, rho_re, r_bs0, ecc=BS_ECCENT):
    """Shortest distance from (x, rho) to the model surface, signed positive outside it."""
    L = r_bs0 * (1.0 + ecc)
    r = L / (1.0 + ecc * np.cos(_theta))
    d = np.hypot(r * np.cos(_theta) - x_re, r * np.sin(_theta) - rho_re)
    r_point = np.hypot(x_re, rho_re)
    r_surf = L / (1.0 + ecc * np.cos(np.arctan2(rho_re, x_re)))
    return d.min() * (1.0 if r_point > r_surf else -1.0)


def omni_at(t0, window_h=OMNI_WINDOW_H):
    """Nearest hour with all five fields valid, or None if none within window_h."""
    k = np.searchsorted(_omni_epoch, np.datetime64(t0))
    best = None
    for i in (k - 1, k):
        if 0 <= i < len(omni):
            off = (omni.Epoch[i] - t0).total_seconds() / 3600.0
            if best is None or abs(off) < abs(best[1]):
                best = (i, off)
    if best is None or abs(best[1]) > window_h:
        return None
    i, off = best
    r = omni.loc[i]
    return dict(Pdyn=float(r.Flow_pressure), Bz=float(r.Bz_GSM),
                Ma=float(r.Alfven_mach_number), omni_epoch=r.Epoch, offset_h=off)

## 3. Positions and distances, one row per crossing

In [ ]:
# linear interpolation of the 10-minute ephemeris onto the crossing times
_t_eph = eph.index.values.astype("datetime64[ns]").astype("int64")
_xyz = eph[["GSM_X [km]", "GSM_Y [km]", "GSM_Z [km]"]].to_numpy() / RE_KM
_t_cross = x["time"].values.astype("datetime64[ns]").astype("int64")
_pos = np.column_stack([np.interp(_t_cross, _t_eph, _xyz[:, j]) for j in range(3)])


def leg_parameters(pass_id, sc_direction):
    """The single parameter set the rendering used for this leg, or None if it was skipped."""
    key = (pass_id, sc_direction)
    if key not in par_ok.index:
        return None
    r = par_ok.loc[key]
    return dict(Pdyn=float(r.Pdyn), Bz=float(r.Bz), Ma=float(r.Ma),
                omni_epoch=pd.to_datetime(r.omni_epoch), offset_h=float(r.offset_h))


rows = []
for (_, cr), (X, Y, Z) in zip(x.iterrows(), _pos):
    pass_id, sc = int(cr["pass"]), cr["SC direction"]
    rho = float(np.hypot(Y, Z))
    rec = dict(pass_id=pass_id, sc_direction=sc, time=cr["time"], crossing_direction=cr["direction"],
               x=X, y=Y, z=Z, rho=rho, azimuth=float(np.degrees(np.arctan2(Z, Y))))

    p = omni_at(cr["time"]) if PARAM_SOURCE == "crossing" else leg_parameters(pass_id, sc)
    if p is None:
        rec["status"] = "no OMNI within window"
        rows.append(rec)
        continue

    r0, alpha = shue98_r0_alpha(p["Pdyn"], p["Bz"])
    delta = standoff(r0, p["Ma"])
    r_bs0 = r0 + delta
    rho_model = bowshock_rho_at_x(X, r_bs0)
    rec.update(status="ok", Pdyn=p["Pdyn"], Bz=p["Bz"], Ma=p["Ma"],
               omni_epoch=p["omni_epoch"], offset_h=p["offset_h"],
               r0_mp=r0, alpha=alpha, delta=delta, r_bs0=r_bs0,
               rho_model=rho_model,
               d_fixed_x=abs(rho - rho_model) if np.isfinite(rho_model) else np.nan,
               d_min_signed=bowshock_min_distance(X, rho, r_bs0))
    rows.append(rec)

cross = pd.DataFrame(rows)
cross["d_min"] = cross.d_min_signed.abs()
cross["distance"] = cross.d_min if DISTANCE_DEF == "min_surface" else cross.d_fixed_x
cross["sunward_of_nose"] = cross.rho_model.isna() & cross.r_bs0.notna()

print(cross.status.value_counts().to_string())
print(f"\nsunward of the model nose: {int(cross.sunward_of_nose.sum())} crossings")
sel = cross[(cross.x >= X_MIN) & cross.distance.notna()].copy()
print(f"used in the figure: {len(sel)} crossings, {sel.pass_id.nunique()} passes, "
      f"{sel.groupby(['pass_id', 'sc_direction']).ngroups} legs")

## 4. Both choices side by side, so the decision is made on numbers

In [ ]:
def summarise(frame, column, label):
    v = frame[column].dropna()
    return dict(choice=label, N=len(v), passes=frame.loc[v.index, "pass_id"].nunique(),
                median=v.median(), q1=v.quantile(0.25), q3=v.quantile(0.75), mean=v.mean(),
                within_5=100 * (v <= 5).mean(), within_10=100 * (v <= 10).mean(),
                n_over_10=int((v > 10).sum()), max=v.max())

day = cross[cross.x >= X_MIN]
comp = pd.DataFrame([summarise(day, "d_fixed_x", "fixed_x"),
                     summarise(day, "d_min", "min_surface")]).set_index("choice")
print("current PARAM_SOURCE =", PARAM_SOURCE)
print(comp.round(2).to_string())

over = day[day[("d_min" if DISTANCE_DEF == "min_surface" else "d_fixed_x")] > 10]
print(f"\nbeyond 10 R_E: {len(over)} crossings from passes {sorted(over.pass_id.unique())}")
if len(over):
    print(over[["pass_id", "sc_direction", "time", "x", "rho", "Ma", "r_bs0", "distance"]]
          .round(2).to_string(index=False))

print("\nsign of the separation (min_surface only): "
      f"{100 * (day.d_min_signed.dropna() > 0).mean():.0f}% of the crossings lie outside the model "
      f"surface, median {day.d_min_signed.median():+.2f} R_E")

## 5. The numbers the reply needs, printed ready to paste

In [ ]:
d = sel.distance
n_pass = sel.pass_id.nunique()
over = sel[sel.distance > 10]

print("=" * 78)
print(f"definition: {DISTANCE_DEF}   parameters per {PARAM_SOURCE}   selection x_GSM >= {X_MIN}")
print("=" * 78)
print(f"crossings used          {len(sel)} of {len(cross)}, from {n_pass} passes")
print(f"median distance         {d.median():.2f} R_E   (IQR {d.quantile(.25):.2f}-{d.quantile(.75):.2f})")
print(f"arithmetic mean         {d.mean():.2f} R_E")
print(f"within 5 R_E            {100 * (d <= 5).mean():.0f}%")
print(f"within 10 R_E           {100 * (d <= 10).mean():.0f}%")
print(f"beyond 10 R_E           {len(over)} crossings from {over.pass_id.nunique()} passes "
      f"{sorted(over.pass_id.unique())}")
print(f"largest separation      {d.max():.2f} R_E")

print("\nper azimuthal bin")
sel["azimuth_bin"] = pd.cut(sel.azimuth, AZIMUTH_EDGES)
stats = (sel.groupby("azimuth_bin", observed=False)
            .agg(n=("distance", "size"), passes=("pass_id", "nunique"),
                 median=("distance", "median"),
                 q1=("distance", lambda s: s.quantile(0.25)),
                 q3=("distance", lambda s: s.quantile(0.75))))
print(stats.round(2).to_string())
empty = stats.index[stats["n"] < MIN_BIN_COUNT].tolist()
print(f"\nbins below the drawing threshold: {len(empty)}")
print(f"most populated bin {int(stats['n'].max())} crossings, least populated non-empty "
      f"{int(stats['n'][stats['n'] > 0].min())}")

# ---- the standoff distance each crossing implies, and how it follows the dynamic pressure --------
# Scaling the conic r = r_bs0 (1+e)/(1 + e cos(theta)) so that it passes through a crossing at
# radius r and polar angle theta from +x gives r_bs0_obs = r (1 + e cos(theta)) / (1 + e).
radius = np.sqrt(sel.x ** 2 + sel.y ** 2 + sel.z ** 2)
sel["r_bs0_obs"] = radius * (1.0 + BS_ECCENT * sel.x / radius) / (1.0 + BS_ECCENT)
over = sel[sel.distance > 10]          # rebuilt, so panel (d) can mark it too

pressure_edges   = np.unique(np.quantile(sel.Pdyn, np.linspace(0, 1, N_PRESSURE_BINS + 1)))
pressure_centers = np.sqrt(pressure_edges[:-1] * pressure_edges[1:])
sel["pressure_bin"] = pd.cut(sel.Pdyn, pressure_edges, include_lowest=True)
grouped = sel.groupby("pressure_bin", observed=False)
pressure_stats = grouped.agg(n=("r_bs0_obs", "size"), passes=("pass_id", "nunique"),
                             median=("r_bs0_obs", "median"),
                             q1=("r_bs0_obs", lambda s: s.quantile(0.25)),
                             q3=("r_bs0_obs", lambda s: s.quantile(0.75)),
                             model=("r_bs0", "median"))
pressure_stats["ratio"] = pressure_stats["median"] / pressure_stats["model"]

log_p = np.log10(sel.Pdyn)
fit_slope   = np.polyfit(log_p, np.log10(sel.r_bs0_obs), 1)[0]
model_slope = np.polyfit(log_p, np.log10(sel.r_bs0), 1)[0]
rank_rho    = np.corrcoef(sel.Pdyn.rank(), sel.r_bs0_obs.rank())[0, 1]
ratio       = sel.r_bs0_obs / sel.r_bs0

print("\nimplied standoff against the dynamic pressure")
print(f"power-law fit           P^{fit_slope:+.3f}   model P^{model_slope:+.3f}   "
      f"gas dynamic P^-0.167")
print(f"Spearman rho            {rank_rho:+.3f}")
print(f"observed / model        median {ratio.median():.3f} "
      f"(IQR {ratio.quantile(.25):.3f}-{ratio.quantile(.75):.3f})")
print(pressure_stats.round(2).to_string())

## 6. Drawing helpers — one per element, every visual property passed at the point of use

In [ ]:
def new_figure(width, height, dpi, width_ratios, cbar_width, wspace, hspace):
    """Two by two, with the colorbar column attached to the bottom row so all four panels align."""
    fig = plt.figure(figsize=(width, height), dpi=dpi)
    gs = fig.add_gridspec(2, 3, width_ratios=(*width_ratios, cbar_width),
                          wspace=wspace, hspace=hspace)
    axes = [fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1]),
            fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])]
    return fig, axes, fig.add_subplot(gs[1, 2])


def add_histogram(ax, values, bins, facecolor, edgecolor, linewidth, alpha):
    return ax.hist(values, bins=bins, color=facecolor, edgecolor=edgecolor,
                   linewidth=linewidth, alpha=alpha)


def add_vline(ax, xvalue, color, linestyle, linewidth, label):
    return ax.axvline(xvalue, color=color, linestyle=linestyle, linewidth=linewidth, label=label)


def add_text(ax, xpos, ypos, text, fontsize, color, ha, va, rotation=0):
    return ax.text(xpos, ypos, text, fontsize=fontsize, color=color,
                   ha=ha, va=va, rotation=rotation)


def add_text_above(ax, xpos, yfraction, text, fontsize, color, ha, va):
    """Text at a data x and a fixed fraction of the panel height, so a log axis keeps it in place."""
    return ax.text(xpos, yfraction, text, transform=ax.get_xaxis_transform(),
                   fontsize=fontsize, color=color, ha=ha, va=va)


def add_panel_label(ax, text, xpos, ypos, fontsize, color):
    return ax.text(xpos, ypos, text, transform=ax.transAxes, fontsize=fontsize, color=color,
                   ha="left", va="bottom", fontweight="bold")


def add_log_xticks(ax, values, fontsize):
    """Plain decimal labels on a log axis, in place of the default powers of ten."""
    ax.set_xticks(values)
    ax.set_xticks([], minor=True)
    ax.set_xticklabels([f"{v:g}" for v in values], fontsize=fontsize)
    return ax


def add_axis_break(ax, ylow, yhigh, tilt, facecolor, color, linewidth):
    """A tilted double line across the panel, marking where the y scale jumps."""
    ax.axhspan(ylow, yhigh, facecolor=facecolor, edgecolor="none", zorder=4)
    for y in (ylow, yhigh):
        ax.plot((0, 1), (y, y + tilt), transform=ax.get_yaxis_transform(), color=color,
                linewidth=linewidth, clip_on=False, zorder=5)
    return ax


def add_broken_yticks(ax, values, extra_position, extra_label, fontsize):
    """Ticks below the cut, plus one standing for the value drawn above it."""
    ax.set_yticks(list(values) + [extra_position])
    ax.set_yticklabels([f"{v:g}" for v in values] + [extra_label], fontsize=fontsize)
    return ax


def add_median_iqr(ax, centers, median, q1, q3, color, marker, markersize, linewidth, capsize,
                   linestyle="none"):
    yerr = np.vstack([median - q1, q3 - median])
    return ax.errorbar(centers, median, yerr=yerr, color=color, marker=marker,
                       markersize=markersize, linewidth=linewidth, capsize=capsize,
                       linestyle=linestyle)


def add_line(ax, xvalues, yvalues, color, linestyle, linewidth, marker, markersize, label):
    return ax.plot(xvalues, yvalues, color=color, linestyle=linestyle, linewidth=linewidth,
                   marker=marker, markersize=markersize, label=label)


def add_scatter(ax, xvalues, yvalues, color_values, cmap, size, alpha, edgecolor, edgewidth):
    return ax.scatter(xvalues, yvalues, c=color_values, cmap=cmap, s=size, alpha=alpha,
                      edgecolors=edgecolor, linewidths=edgewidth)


def add_marker(ax, xvalues, yvalues, color, size, marker, linewidth, label):
    return ax.scatter(xvalues, yvalues, color=color, s=size, marker=marker,
                      linewidths=linewidth, label=label)


def add_colorbar(fig, mappable, cax, label, labelsize, ticksize):
    cb = fig.colorbar(mappable, cax=cax)
    cb.set_label(label, fontsize=labelsize)
    cb.ax.tick_params(labelsize=ticksize)
    return cb


def style_axes(ax, xlabel, ylabel, title, labelsize, titlesize, ticksize,
               xscale="linear", yscale="linear",
               xlim=None, ylim=None, xticks=None, grid=True,
               grid_color=GRID_COLOR, grid_alpha=GRID_ALPHA, grid_width=GRID_WIDTH):
    ax.set_xlabel(xlabel, fontsize=labelsize)
    ax.set_ylabel(ylabel, fontsize=labelsize)
    if title:
        ax.set_title(title, fontsize=titlesize)
    ax.set_xscale(xscale)
    ax.set_yscale(yscale)
    ax.tick_params(labelsize=ticksize)
    if xlim:
        ax.set_xlim(*xlim)
    if ylim:
        ax.set_ylim(*ylim)
    if xticks is not None:
        ax.set_xticks(xticks)
    if grid:
        ax.grid(True, color=grid_color, alpha=grid_alpha, linewidth=grid_width)
        ax.set_axisbelow(True)
    return ax

## 7. The figure

In [ ]:
fig, (ax_hist, ax_bin, ax_scatter, ax_press), cax = new_figure(
    FIG_W, FIG_H, FIG_DPI, PANEL_WIDTHS, CBAR_WIDTH, PANEL_WSPACE, PANEL_HSPACE)

# ---- (a) distribution of the distances ------------------------------------------------------------
add_histogram(ax_hist, sel.distance, HIST_BINS, HIST_FACE, HIST_EDGE, HIST_EDGEWIDTH, HIST_ALPHA)
add_vline(ax_hist, sel.distance.median(), MEDIAN_LINE_COLOR, MEDIAN_LINE_STYLE, MEDIAN_LINE_WIDTH,
          f"median {sel.distance.median():.2f} $R_\\oplus$")
style_axes(ax_hist, "distance from the model surface [$R_\\oplus$]", "crossings", None,
           LABEL_SIZE, TITLE_SIZE, TICK_SIZE, xlim=(0, HIST_BINS[-1]))
ax_hist.legend(fontsize=TICK_SIZE, frameon=False)

# ---- (b) median and IQR per azimuthal bin ---------------------------------------------------------
centers = 0.5 * (AZIMUTH_EDGES[:-1] + AZIMUTH_EDGES[1:])
filled = (stats["n"] >= MIN_BIN_COUNT).values
add_median_iqr(ax_bin, centers[filled], stats["median"].values[filled],
               stats["q1"].values[filled], stats["q3"].values[filled],
               IQR_COLOR, IQR_MARKER, IQR_MARKERSIZE, IQR_LINEWIDTH, IQR_CAPSIZE)

y_annot = np.nanmax(stats["q3"].values[filled]) * 1.08
for c, n, p in zip(centers[filled], stats["n"].values[filled], stats["passes"].values[filled]):
    add_text(ax_bin, c, y_annot, f"{int(n)}\n{int(p)}p", ANNOT_FONTSIZE, ANNOT_COLOR,
             "center", "bottom")
for c in centers[~filled]:
    add_text(ax_bin, c, y_annot * 0.5, "empty", EMPTY_FONTSIZE, EMPTY_COLOR, "center", "center",
             rotation=90)

style_axes(ax_bin, "azimuth about the $x$ axis [deg]", "distance [$R_\\oplus$]", None,
           LABEL_SIZE, TITLE_SIZE, TICK_SIZE, xlim=(-180, 180),
           ylim=(0, y_annot * AZIMUTH_ANNOT_HEADROOM), xticks=np.arange(-180, 181, 60))

# ---- (c) every crossing against azimuth, coloured by pass -----------------------------------------
points = add_scatter(ax_scatter, sel.azimuth, sel.distance, sel.pass_id, SCATTER_CMAP,
                     SCATTER_SIZE, SCATTER_ALPHA, SCATTER_EDGE, SCATTER_EDGEWIDTH)
style_axes(ax_scatter, "azimuth about the $x$ axis [deg]", "distance [$R_\\oplus$]", None,
           LABEL_SIZE, TITLE_SIZE, TICK_SIZE, xlim=(-180, 180),
           ylim=(0, SCATTER_Y_TOP), xticks=np.arange(-180, 181, 60))
if len(over):
    # the scale jumps above BREAK_LOW, and the crossing beyond it is drawn above the jump at a
    # tick carrying its real value; the marker has no label, the caption identifies it
    add_axis_break(ax_scatter, BREAK_LOW, BREAK_HIGH, BREAK_TILT,
                   BREAK_FACECOLOR, BREAK_COLOR, BREAK_WIDTH)
    add_broken_yticks(ax_scatter, SCATTER_YTICKS, OUTLIER_DRAWN_AT,
                      f"{over.distance.iloc[0]:.0f}", TICK_SIZE)
    add_marker(ax_scatter, over.azimuth, [OUTLIER_DRAWN_AT] * len(over), OUTLIER_COLOR,
               OUTLIER_SIZE, OUTLIER_MARKER, OUTLIER_LINEWIDTH, "_nolegend_")

# ---- (d) the standoff each crossing implies, against the dynamic pressure -------------------------
add_scatter(ax_press, sel.Pdyn, sel.r_bs0_obs, sel.pass_id, SCATTER_CMAP,
            SCATTER_SIZE, SCATTER_ALPHA, SCATTER_EDGE, SCATTER_EDGEWIDTH)
add_median_iqr(ax_press, pressure_centers, pressure_stats["median"].values,
               pressure_stats["q1"].values, pressure_stats["q3"].values,
               IQR_COLOR, IQR_MARKER, IQR_MARKERSIZE, IQR_LINEWIDTH, IQR_CAPSIZE, linestyle="-")
add_line(ax_press, pressure_centers, pressure_stats["model"].values, MODEL_COLOR, MODEL_STYLE,
         MODEL_WIDTH, MODEL_MARKER, MODEL_MARKERSIZE, "model")
if len(over):
    add_marker(ax_press, over.Pdyn, over.r_bs0_obs, OUTLIER_COLOR, OUTLIER_SIZE,
               OUTLIER_MARKER, OUTLIER_LINEWIDTH, "_nolegend_")

style_axes(ax_press, "OMNI dynamic pressure [nPa]", "implied standoff $r_{bs0}$ [$R_\\oplus$]", None,
           LABEL_SIZE, TITLE_SIZE, TICK_SIZE,
           xscale=PRESSURE_XSCALE, yscale=PRESSURE_YSCALE,
           xlim=(sel.Pdyn.min() * PRESSURE_X_LOW, sel.Pdyn.max() * PRESSURE_X_HIGH),
           ylim=(sel.r_bs0_obs.min() * PRESSURE_Y_LOW, sel.r_bs0_obs.max() * PRESSURE_Y_HIGH))
add_log_xticks(ax_press, PRESSURE_XTICKS, TICK_SIZE)
for i, (c, n, p) in enumerate(zip(pressure_centers, pressure_stats["n"].values,
                                  pressure_stats["passes"].values)):
    add_text_above(ax_press, c, PRESSURE_ANNOT_YFRAC[i % 2], f"{int(n)}\n{int(p)}p",
                   ANNOT_FONTSIZE, ANNOT_COLOR, "center", "bottom")
ax_press.legend(fontsize=TICK_SIZE, frameon=False, loc=PRESSURE_LEGEND_LOC)

# ---- one colorbar, for the two panels that colour by pass -----------------------------------------
add_colorbar(fig, points, cax, "pass number", LABEL_SIZE, TICK_SIZE)

for ax, label in zip((ax_hist, ax_bin, ax_scatter, ax_press), PANEL_LABELS):
    add_panel_label(ax, label, PANEL_LABEL_X, PANEL_LABEL_Y, PANEL_LABEL_SIZE, PANEL_LABEL_COLOR)

fig.savefig(OUT_FIG, dpi=FIG_DPI, bbox_inches="tight")
print("wrote", OUT_FIG)
plt.show()

Every label and marker above is placed by its own call. The bin annotations in panel (b) sit at a
data height, the ones in panel (d) at a fraction of the panel height so that the log axis cannot
drag them into the points; to move either, change the number in its own `add_text` or
`add_text_above` call. Nothing positions anything for you.

## 8. Write the per-crossing table

In [ ]:
cols = ["pass_id", "sc_direction", "time", "crossing_direction", "x", "y", "z", "rho", "azimuth",
        "status", "sunward_of_nose", "omni_epoch", "offset_h", "Pdyn", "Bz", "Ma",
        "r0_mp", "alpha", "delta", "r_bs0", "rho_model",
        "d_fixed_x", "d_min_signed", "d_min", "distance"]
cross[cols].to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}  ({len(cross)} rows)")
print("\nBoth distance definitions are in the file, so the figure can be redrawn either way")
print("without recomputing anything.")